# Formativa 2: Regresión logística inicial

**Asignatura:** Estadística Computacional
**Integrantes:** Hernán Saavedra, Lenin Morales y Rodrigo Gómez
**Responsable de este notebook:** Lenin Morales
**Dataset:** IBM HR Employee Attrition

## Objetivo

Ajustar un modelo de regresión logística que estime la probabilidad de
salida de un empleado (Attrition = 1) a partir de los cuatro predictores
preparados en `F2_Hernan_Preparacion.ipynb`: OverTime, MonthlyIncome,
YearsAtCompany y JobSatisfaction.

Se utiliza `statsmodels` para obtener la inferencia del modelo
(coeficientes, errores estándar, valores p e intervalos), se calculan
odds ratios e índices VIF y se generan probabilidades para el conjunto
de prueba.

Siguiendo la guía docente, este modelo inicial **no aplica ponderación
de clases**, de modo que se puedan observar sus limitaciones frente al
desbalance de Attrition. La ponderación se incorporará en S3.

El conjunto de prueba solo se utiliza para generar probabilidades; su
variable objetivo no interviene en este notebook.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Localizar la raíz del repositorio desde la ubicación del notebook.
RAIZ = next(
    (
        carpeta
        for carpeta in [Path.cwd(), *Path.cwd().parents]
        if (carpeta / "datos" / "WA_Fn-UseC_-HR-Employee-Attrition.csv").is_file()
    ),
    None,
)

if RAIZ is None:
    raise FileNotFoundError("No se encontró la raíz del repositorio.")

DIRECTORIO_F2 = RAIZ / "Formativa2"
CARPETA_DATOS = DIRECTORIO_F2 / "datos"
CARPETA_RESULTADOS = DIRECTORIO_F2 / "resultados"
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("statsmodels:", statsmodels.__version__)
print("Raíz del proyecto:", RAIZ.name)

Python: 3.13.17
statsmodels: 0.15.0
Raíz del proyecto: repo


## 1. Carga de los insumos preparados

Se cargan `X_train.csv`, `y_train.csv` y `X_test.csv` utilizando
EmployeeNumber como índice. El identificador permite comprobar que cada
fila de predictores corresponde a la misma persona en la variable
objetivo; no se utiliza como predictor.

Se verifican además los predictores esperados según
`configuracion_preparacion.json`, la ausencia de faltantes y la
codificación binaria de OverTime y Attrition.

In [2]:
X_train = pd.read_csv(CARPETA_DATOS / "X_train.csv", index_col="EmployeeNumber")
y_train = pd.read_csv(CARPETA_DATOS / "y_train.csv", index_col="EmployeeNumber")["Attrition"]
X_test = pd.read_csv(CARPETA_DATOS / "X_test.csv", index_col="EmployeeNumber")

configuracion = json.loads(
    (CARPETA_RESULTADOS / "configuracion_preparacion.json").read_text(encoding="utf-8")
)
PREDICTORES = configuracion["predictores"]

# Comprobaciones de consistencia con la preparación.
assert list(X_train.columns) == PREDICTORES
assert list(X_test.columns) == PREDICTORES
assert X_train.index.equals(y_train.index), "X_train e y_train no coinciden."
assert X_train.index.is_unique and X_test.index.is_unique
assert set(X_train.index).isdisjoint(X_test.index)
assert not X_train.isna().any().any() and not X_test.isna().any().any()
assert set(X_train["OverTime"].unique()) <= {0, 1}
assert set(y_train.unique()) == {0, 1}
assert len(X_train) == configuracion["n_entrenamiento"]
assert len(X_test) == configuracion["n_prueba"]

resumen_objetivo = (
    y_train.value_counts()
    .rename_axis("Attrition")
    .to_frame("Cantidad")
)
resumen_objetivo["Porcentaje"] = resumen_objetivo["Cantidad"] / len(y_train) * 100

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
display(resumen_objetivo.round(2))

Entrenamiento: (1029, 4)
Prueba: (441, 4)


,Cantidad,Porcentaje
Attrition,,
0,863,83.87
1,166,16.13


La clase positiva (salida) representa cerca del 16 % del entrenamiento.
Este desbalance es relevante: un modelo sin ponderación tenderá a
estimar probabilidades bajas para la mayoría de los empleados.

## 2. Ajuste del modelo logístico

El modelo estima el logaritmo de las odds de salida como una
combinación lineal de los predictores:

$$
\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1\,\text{OverTime}
+ \beta_2\,\text{MonthlyIncome} + \beta_3\,\text{YearsAtCompany}
+ \beta_4\,\text{JobSatisfaction}
$$

donde $p$ es la probabilidad de salida y $\beta_0$ es el intercepto.
`statsmodels` no agrega el intercepto automáticamente, por lo que se
incorpora con `sm.add_constant`. Los parámetros se estiman por máxima
verosimilitud.

In [3]:
X_train_const = sm.add_constant(X_train, has_constant="add")

modelo = sm.Logit(y_train, X_train_const)
resultado = modelo.fit(disp=False)

assert resultado.mle_retvals["converged"], "El modelo no convergió."

print(resultado.summary())

                           Logit Regression Results                           
Dep. Variable:              Attrition   No. Observations:                 1029
Model:                          Logit   Df Residuals:                     1024
Method:                           MLE   Df Model:                            4
Date:                Thu, 22 Oct 2026   Pseudo R-squ.:                  0.1160
Time:                        23:34:43   Log-Likelihood:                -401.93
converged:                       True   LL-Null:                       -454.67
Covariance Type:            nonrobust   LLR p-value:                 6.712e-22
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -2.3234      0.135    -17.261      0.000      -2.587      -2.060
OverTime            1.2840      0.182      7.063      0.000       0.928       1.640
MonthlyIncome      -0.4301      

## 3. Coeficientes, valores p, odds ratios e intervalos del 95 %

Cada odds ratio (OR) se obtiene como $e^{\beta}$ y sus intervalos del
95 % se obtienen exponenciando los límites del intervalo del
coeficiente. Un OR mayor que 1 indica que el predictor se asocia con
mayores odds de salida; un OR menor que 1, con menores odds. Si el
intervalo del OR no contiene el 1, la asociación es significativa al
5 %.

In [4]:
intervalos = resultado.conf_int(alpha=0.05)
intervalos.columns = ["IC 2,5 %", "IC 97,5 %"]

tabla_coeficientes = pd.DataFrame({
    "Coeficiente": resultado.params,
    "Error estándar": resultado.bse,
    "z": resultado.tvalues,
    "Valor p": resultado.pvalues,
    "IC coef. 2,5 %": intervalos["IC 2,5 %"],
    "IC coef. 97,5 %": intervalos["IC 97,5 %"],
    "Odds ratio": np.exp(resultado.params),
    "IC OR 2,5 %": np.exp(intervalos["IC 2,5 %"]),
    "IC OR 97,5 %": np.exp(intervalos["IC 97,5 %"]),
})
tabla_coeficientes.index.name = "Término"

tabla_coeficientes.to_csv(CARPETA_RESULTADOS / "lenin_logistica_coeficientes.csv")

indicadores_ajuste = pd.Series({
    "Observaciones": int(resultado.nobs),
    "Log-verosimilitud": resultado.llf,
    "Log-verosimilitud nula": resultado.llnull,
    "Pseudo R² de McFadden": resultado.prsquared,
    "Estadístico LR": resultado.llr,
    "Valor p LR": resultado.llr_pvalue,
    "AIC": resultado.aic,
}, name="Valor")

display(tabla_coeficientes.round(4))
display(indicadores_ajuste.to_frame().round(4))

,Coeficiente,Error estándar,z,Valor p,"IC coef. 2,5 %","IC coef. 97,5 %",Odds ratio,"IC OR 2,5 %","IC OR 97,5 %"
Término,,,,,,,,,
const,-2.3234,0.1346,-17.2612,0.0000,-2.5872,-2.0596,0.0979,0.0752,0.1275
OverTime,1.2840,0.1818,7.0632,0.0000,0.9277,1.6403,3.6112,2.5288,5.1569
MonthlyIncome,-0.4301,0.1281,-3.3582,0.0008,-0.6811,-0.1791,0.6505,0.5061,0.8360
YearsAtCompany,-0.4958,0.1350,-3.6724,0.0002,-0.7605,-0.2312,0.6091,0.4674,0.7936
JobSatisfaction,-0.3072,0.0881,-3.4847,0.0005,-0.4799,-0.1344,0.7355,0.6188,0.8742


,Valor
Observaciones,1029.0000
Log-verosimilitud,-401.9317
Log-verosimilitud nula,-454.6688
Pseudo R² de McFadden,0.1160
Estadístico LR,105.4743
Valor p LR,0.0000
AIC,813.8634


## 4. Multicolinealidad mediante VIF

El factor de inflación de varianza (VIF) de un predictor indica cuánto
aumenta la varianza de su coeficiente por su correlación con los demás
predictores: $\text{VIF}_j = 1/(1-R_j^2)$, donde $R_j^2$ es el
coeficiente de determinación de regresionar el predictor $j$ sobre el
resto.

Se calcula con la matriz que incluye el intercepto, porque sin él el VIF
de variables no centradas queda sobreestimado. Se interpretan solo los
VIF de los predictores; el del intercepto no tiene una lectura
sustantiva. Como referencia habitual, valores cercanos a 1 indican
ausencia de colinealidad, y valores sobre 5 o 10 requieren revisión.

In [5]:
matriz_vif = X_train_const.astype(float)

tabla_vif = pd.DataFrame({
    "Término": matriz_vif.columns,
    "VIF": [
        variance_inflation_factor(matriz_vif.to_numpy(), i)
        for i in range(matriz_vif.shape[1])
    ],
}).set_index("Término")

tabla_vif["Interpretación"] = np.where(
    tabla_vif.index == "const",
    "Intercepto: no se interpreta",
    np.select(
        [tabla_vif["VIF"] < 2, tabla_vif["VIF"] < 5],
        ["Colinealidad despreciable", "Colinealidad moderada"],
        default="Colinealidad alta: revisar",
    ),
)

tabla_vif.to_csv(CARPETA_RESULTADOS / "lenin_logistica_vif.csv")

assert (tabla_vif.drop(index="const")["VIF"] < 5).all()
display(tabla_vif.round(3))

,VIF,Interpretación
Término,,
const,1.000,Intercepto: no se interpreta
OverTime,1.003,Colinealidad despreciable
MonthlyIncome,1.301,Colinealidad despreciable
YearsAtCompany,1.299,Colinealidad despreciable
JobSatisfaction,1.000,Colinealidad despreciable


Todos los VIF de los predictores están cerca de 1. Los valores
levemente mayores de MonthlyIncome y YearsAtCompany (alrededor de 1,3)
son coherentes con la correlación positiva entre ingreso y antigüedad
observada en S1 y validada en S2, pero no comprometen la estabilidad de
los coeficientes. La exclusión de JobLevel, TotalWorkingYears y Age en la
preparación contribuyó a mantener baja la redundancia.

El VIF del intercepto resulta cercano a 1 porque las variables numéricas
están centradas en su media de entrenamiento.

## 5. Interpretación de los odds ratios

MonthlyIncome, YearsAtCompany y JobSatisfaction fueron estandarizadas con
la media y desviación del entrenamiento. Por ello, su OR representa el
cambio multiplicativo en las odds de salida ante un aumento de **una
desviación estándar**, manteniendo constantes los demás predictores. La
tabla siguiente traduce esa desviación a las unidades originales.

OverTime no fue estandarizado (0 = No, 1 = Sí), por lo que su OR compara
directamente las odds de salida de quienes hacen horas extra frente a
quienes no las hacen.

El OR se refiere a **odds**, no a probabilidades: un OR de 0,65 no
significa que la probabilidad baje un 35 %, sino que la razón
$p/(1-p)$ se multiplica por 0,65.

In [6]:
parametros_escalado = pd.read_csv(
    CARPETA_RESULTADOS / "parametros_escalado.csv", index_col="Variable"
)

unidad = pd.Series({
    "OverTime": "Sí frente a No",
    "MonthlyIncome": "+1 DE (≈ {:,.0f} de ingreso mensual)",
    "YearsAtCompany": "+1 DE (≈ {:,.1f} años de antigüedad)",
    "JobSatisfaction": "+1 DE (≈ {:,.2f} niveles de satisfacción)",
})
for variable in parametros_escalado.index:
    unidad[variable] = unidad[variable].format(
        parametros_escalado.loc[variable, "Desviación entrenamiento"]
    )

interpretacion = tabla_coeficientes.loc[
    PREDICTORES, ["Odds ratio", "IC OR 2,5 %", "IC OR 97,5 %", "Valor p"]
].copy()
interpretacion.insert(0, "Comparación", unidad[PREDICTORES])
interpretacion["Cambio en odds (%)"] = (interpretacion["Odds ratio"] - 1) * 100
interpretacion["Cambio IC 2,5 % (%)"] = (interpretacion["IC OR 2,5 %"] - 1) * 100
interpretacion["Cambio IC 97,5 % (%)"] = (interpretacion["IC OR 97,5 %"] - 1) * 100

display(interpretacion.round(3))

,Comparación,Odds ratio,"IC OR 2,5 %","IC OR 97,5 %",Valor p,Cambio en odds (%),"Cambio IC 2,5 % (%)","Cambio IC 97,5 % (%)"
Término,,,,,,,,
OverTime,Sí frente a No,3.611,2.529,5.157,0.000,261.119,152.876,415.695
MonthlyIncome,"+1 DE (≈ 4,404 de ingreso mensual)",0.650,0.506,0.836,0.001,-34.954,-49.393,-16.395
YearsAtCompany,+1 DE (≈ 5.9 años de antigüedad),0.609,0.467,0.794,0.000,-39.094,-53.256,-20.643
JobSatisfaction,+1 DE (≈ 1.11 niveles de satisfacción),0.736,0.619,0.874,0.000,-26.446,-38.116,-12.575


**Lectura de los resultados** (todos los efectos manteniendo constantes
los demás predictores):

- **OverTime:** quienes hacen horas extra presentan odds de salida
  aproximadamente 3,6 veces las de quienes no las hacen (IC 95 %: 2,5 a
  5,2). Es el predictor de mayor magnitud y es coherente con S1, donde la
  tasa de salida fue de 30,53 % con horas extra frente a 10,44 % sin
  ellas.
- **MonthlyIncome:** un aumento de una desviación estándar del ingreso
  (cerca de 4.400) se asocia con odds de salida un 35 % menores
  (OR ≈ 0,65; IC 95 %: 0,51 a 0,84). La dirección concuerda con los
  intervalos bootstrap de S2.
- **YearsAtCompany:** un aumento de una desviación estándar de antigüedad
  (cerca de 6 años) se asocia con odds de salida un 39 % menores
  (OR ≈ 0,61; IC 95 %: 0,47 a 0,79).
- **JobSatisfaction:** un aumento de una desviación estándar (cerca de un
  nivel de la escala 1 a 4) se asocia con odds de salida un 26 % menores
  (OR ≈ 0,74; IC 95 %: 0,62 a 0,87). Esta lectura supone un efecto lineal
  de la escala ordinal sobre el logit, supuesto que se revisará en S3.

Los cuatro predictores son significativos al 5 % y ningún intervalo del
OR contiene el 1. El intercepto representa las odds de salida de un
empleado sin horas extra y con valores medios en las variables
numéricas (OR ≈ 0,10, equivalente a una probabilidad cercana a 9 %).

El pseudo R² de McFadden (≈ 0,12) indica que el modelo mejora
claramente al modelo nulo (prueba LR con valor p muy bajo), aunque
gran parte de la variabilidad de Attrition queda sin explicar con solo
cuatro predictores. Estas asociaciones no implican causalidad y los
datos son sintéticos.

## 6. Probabilidades sobre el conjunto de prueba

Se aplican los coeficientes estimados a `X_test`, que fue imputado y
estandarizado con parámetros del entrenamiento. Las probabilidades se
guardan junto a EmployeeNumber en
`Formativa2/resultados/probabilidades_prueba.csv` para la evaluación
posterior del equipo.

In [7]:
X_test_const = sm.add_constant(X_test, has_constant="add")
assert list(X_test_const.columns) == list(X_train_const.columns)

probabilidades = pd.DataFrame({
    "Probabilidad_salida": resultado.predict(X_test_const)
}, index=X_test.index)
probabilidades.index.name = "EmployeeNumber"

assert probabilidades["Probabilidad_salida"].between(0, 1).all()
assert probabilidades.index.equals(X_test.index)

RUTA_PROBABILIDADES = CARPETA_RESULTADOS / "probabilidades_prueba.csv"
probabilidades.to_csv(RUTA_PROBABILIDADES)

umbral = configuracion["umbral_inicial"]
resumen_probabilidades = probabilidades["Probabilidad_salida"].describe()
resumen_probabilidades["Casos con p ≥ umbral"] = (
    probabilidades["Probabilidad_salida"] >= umbral
).sum()
resumen_probabilidades["Prevalencia en entrenamiento"] = y_train.mean()

print("Archivo guardado en:", RUTA_PROBABILIDADES.relative_to(RAIZ))
display(probabilidades.head())
display(resumen_probabilidades.to_frame("Valor").round(4))

Archivo guardado en: Formativa2/resultados/probabilidades_prueba.csv


,Probabilidad_salida
EmployeeNumber,
527,0.249116
1161,0.073401
652,0.110536
613,0.046477
1882,0.174066


,Valor
count,441.0000
mean,0.1596
std,0.1236
min,0.0017
25%,0.0712
50%,0.1260
75%,0.2153
max,0.5998
Casos con p ≥ umbral,6.0000
Prevalencia en entrenamiento,0.1613


## 7. Limitaciones del modelo inicial sin ponderación

La probabilidad media estimada en prueba (≈ 16 %) coincide con la
prevalencia de salida del entrenamiento, lo esperable en una regresión
logística con intercepto. Sin embargo, la probabilidad máxima apenas
alcanza cerca de 0,60 y solo unos pocos empleados de prueba superan el
umbral inicial de 0,5, aunque se espera que cerca del 16 % de ellos
salga.

Con ese umbral, el modelo clasificaría casi a todos como permanencia:
tendría una exactitud aparente alta, pero detectaría muy pocos casos
reales de salida. Esta es la limitación principal de no ponderar las
clases cuando la clase positiva es minoritaria.

Para S3 se propone:

1. Incorporar ponderación de clases según la guía docente.
2. Evaluar umbrales distintos de 0,5 con métricas sensibles al
   desbalance (sensibilidad, precisión, F1, curva ROC/PR).
3. Revisar la codificación de JobSatisfaction como variable categórica y
   comparar estrategias de imputación del ingreso.